# Route customer-support requests with the Decisions API

Use a typed `choice` question to route a small batch of public banking-support
messages. Inspect each real response, extract its queue and confidence, and send
refusals or unusable answers to human review. This example only prints routing
decisions; it does not reply to customers, modify accounts, or create tickets.

The demo uses `gpt-6-luna` with `POST /v1/decisions`. The default batch makes
**four billable requests**, one message per queue, from public BANKING77. Increase
`SAMPLES_PER_ROUTE` for a larger demo. There is no calibration/test split,
threshold optimization, model-quality grading, or Promptfoo dependency.


## 1. Configure the demo and load a small message batch

Use Python 3.10 or later. The helpers use only the standard library. For Jupyter,
start the kernel at the repository root or `examples/decisions/`.

Set `SAMPLES_PER_ROUTE = 1` in the first code cell for four API requests, or
`2` for eight. Use **Run All** to load the public messages and display their real
Decisions responses. The notebook asks for your key with hidden input if
`OPENAI_API_KEY` is not already available. Do not put a real key in a code cell
or save it in outputs. Each full run makes new billable requests. Review
[current availability and pricing](https://developers.openai.com/api/docs/guides/decisions#pricing-and-availability).

Live examples come from BANKING77 by Casanueva, Temcinas, Gerz, Henderson, and
Vulić (2020), published by PolyAI under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
See the [source](https://github.com/PolyAI-LDN/task-specific-datasets/tree/57ec275d8078af65b7731c2a98be812d844a6d6b),
[license](https://github.com/PolyAI-LDN/task-specific-datasets/blob/57ec275d8078af65b7731c2a98be812d844a6d6b/LICENSE),
and [paper](https://arxiv.org/abs/2003.04807). We select four source intents,
exclude normalized duplicates/conflicting labels, and deterministically sample
messages from a pinned, checksum-verified CSV. Wording and source labels are
retained; no dataset-author endorsement is implied. Source CSVs are not committed.

Calls = four routes × `SAMPLES_PER_ROUTE`. These are public benchmark examples,
not private customer records. Source labels stay local and are not sent to the model.


In [ ]:
import getpass
import json
import os
from pathlib import Path
import sys
import warnings

example_dir = next((path for path in (Path.cwd(), Path.cwd() / "examples" / "decisions")
                    if (path / "routing.py").is_file()), None)
if example_dir is None:
    raise RuntimeError("Start the kernel in the repository root or examples/decisions")
if str(example_dir) not in sys.path:
    sys.path.insert(0, str(example_dir))

from routing import (API_URL, build_request, parse_prediction,
                     request_prediction, sample_public_messages)

# Set 1 for four API calls, or 2 for eight, then use Run All.
SAMPLES_PER_ROUTE = 1
if isinstance(SAMPLES_PER_ROUTE, bool) or not isinstance(SAMPLES_PER_ROUTE, int) or SAMPLES_PER_ROUTE < 1:
    raise ValueError("Samples per route must be a positive integer")
cases = sample_public_messages(SAMPLES_PER_ROUTE)
print(f"Selected {len(cases)} public messages for billable Decisions requests.")


## 2. Inspect the choice question and request

`build_request(message)` returns the model, input, and one named `choice`
question. Each allowed queue has a description: card-payment fees, app access,
transfer cancellation, or card delivery. The model is instructed to treat
customer text as evidence, not instructions to change the rules.

The API key is supplied separately in the Authorization header. Each request
contains one message and the same question. Inspect the first body below; source
labels are local context only. The Terminal launcher also supports a single custom
message with `--message`.


In [ ]:
print("POST", API_URL)
print(json.dumps(build_request(cases[0]["message"]), indent=2))


## 3. Send the batch and inspect real responses

`request_prediction(message)` reads `OPENAI_API_KEY`, sends one HTTPS request,
and returns its response JSON and elapsed time. The loop makes one call per
message: four by default. It uses a 30-second timeout and no automatic retries.
Inspect HTTP or network errors before repeating a paid run; a request may have
already been processed.

For choice answers, inspect `answers[0].choice`, `confidence`, and per-choice
`probabilities`. Confidence and timing can vary. Request time includes network
and connection setup and is not an inference-only benchmark. Displaying these
few examples does not measure accuracy or establish production performance.


In [ ]:
outputs = []
api_key = None
try:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        # Stop instead of falling back to a prompt that could echo the key.
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            api_key = getpass.getpass("OpenAI API key (hidden): ")
    if not api_key:
        raise RuntimeError("No key supplied; no model requests were made")
    for index, case in enumerate(cases, start=1):
        print(f"Example {index}/{len(cases)}: {case['message']}")
        print("Public dataset label (local context only):", case["source_label"])
        response, latency_ms = request_prediction(case["message"], api_key=api_key)
        print(json.dumps(response, indent=2))
        print(f"Request latency: {latency_ms:.1f} ms")
        outputs.append({"id": case["id"], "message": case["message"], "response": response})
finally:
    api_key = None  # Clear the notebook's key variable after success or failure.


## 4. Validate each answer before using the route

`parse_prediction(response)` accepts one `choice` answer named `route`, checks
its queue against the allowed values, and validates a finite confidence between
zero and one. It returns `(None, None)` for refusals or unusable answers. The code
below maps those cases to `human_review`; it does not execute customer actions.

A refusal can arrive without an HTTP error. Confidence is not a correctness
guarantee, and this demo imposes no unvalidated cutoff. The four choices assume
an in-scope request; a production router also needs a policy for unsupported or
ambiguous messages. Evaluate representative labeled data before choosing
thresholds or enabling downstream actions.


In [ ]:
for item in outputs:
    route, confidence = parse_prediction(item["response"])
    print(json.dumps({"id": item["id"], "route": route or "human_review",
                      "confidence": confidence}, indent=2))


## Adapt the example

The shared [routing helpers](routing.py) contain public-data sampling, question
choices, request construction, and response validation used by this notebook and
the [Terminal runner](run_customer_request_routing.py). Change choices and their
descriptions to match your queue ownership. Keep request text separate from
instructions, and validate answers before use.

This notebook demonstrates routing on a small public sample. It does not train a
model, select thresholds, or report benchmark accuracy. BANKING77 is public and
model exposure during training is unknown. The CSV remains under CC BY 4.0.

See the [README](README.md), [Decisions guide](https://developers.openai.com/api/docs/guides/decisions),
and [API reference](https://developers.openai.com/api/reference/resources/decisions/methods/create).
